# Deterministic forecast evaluation

This notebook shows how to use `ForecastPerformance` to evaluate **deterministic forecasts** with a suite of skill metrics.  



In [ ]:
import pandas as pd
from pathlib import Path
from performance import ForecastPerformance, Results 

test_dataset_path = Path(r'tests\test_datasets')

obs = pd.read_csv(test_dataset_path / 'observed.csv', sep=';', index_col=0, parse_dates=True)
obs.columns = pd.Index(pd.to_timedelta(obs.columns), name='leadtime')
obs.index.name = 'event_datetime'

det_path = test_dataset_path / 'det.parquet'
det_parquet = pd.read_parquet(det_path)
display(det_parquet.head(5))

In [ ]:
results = Results('Model', 'Metric', 'Leadtime')

fp = ForecastPerformance(obs)
fp.add_by_production_date(det_parquet.droplevel('event_datetime').unstack('leadtime'), name='det 1')
fp.add_by_production_date(det_parquet.droplevel('event_datetime').unstack('leadtime') + 2, name='det 2')
fp.add_by_production_date(det_parquet.droplevel('event_datetime').unstack('leadtime') * 1.2, name='det 3')

for model in fp.names():
    print(f'{model}')
    for metric in [fp.KGEprime, fp.NSE, fp.RMSE, fp.MAE, fp.bias, fp.relative_bias, fp.Pearson]:
        for leadtime in det_parquet.index.get_level_values('leadtime').unique():
            results.append(Model=model, Metric=metric.__name__, Leadtime=leadtime,
                        Value=fp.deterministic(metric, model, leadtime=leadtime),
                        )
        
df = results.to_pandas(index=['Metric', 'Model'], columns=['Leadtime'])
display(df.iloc[:, ::6])